
### Key Performance Metrics's
- Speed Monitor
- Emergency Alerts
- Weather Advisory
- City Health Score



### 1. Speed Monitoring 

In [0]:
%sql
CREATE OR REPLACE TABLE smart_city.gold.kpi_speed_monitor AS
SELECT 
    area,
    DATE_TRUNC('hour', timestamp)                           AS hour_window,
    ROUND(AVG(speed), 2)                                    AS avg_speed_kmh,
    ROUND(MAX(speed), 2)                                    AS max_speed_kmh,
    COUNT(DISTINCT deviceId)                                AS active_vehicles,
    COUNT(CASE WHEN speed > 80 THEN 1 END)                  AS overspeeding_count,
    ROUND(COUNT(CASE WHEN speed > 80 THEN 1 END) * 100.0 
          / COUNT(*), 2)                                    AS overspeeding_pct,
    CASE 
        WHEN ROUND(COUNT(CASE WHEN speed > 80 THEN 1 END) * 100.0 
             / COUNT(*), 2) >= 50 THEN 'DANGER'
        WHEN ROUND(COUNT(CASE WHEN speed > 80 THEN 1 END) * 100.0 
             / COUNT(*), 2) >= 25 THEN 'WARNING'
        ELSE 'SAFE'
    END                                                     AS zone_status,
    CURRENT_TIMESTAMP()                                     AS processed_at
FROM smart_city.silver.vehicle_clean
GROUP BY area, DATE_TRUNC('hour', timestamp)
ORDER BY hour_window DESC;

num_affected_rows,num_inserted_rows


#### 2. Emergency Alerts

In [0]:
%sql
CREATE OR REPLACE TABLE smart_city.gold.kpi_emergency_alerts AS
SELECT
    area,
    DATE_TRUNC('hour', timestamp)                           AS hour_window,
    COUNT(*)                                                AS total_incidents,
    COUNT(DISTINCT incidentId)                              AS unique_incidents,

    -- Status breakdown
    COUNT(CASE WHEN status = 'ACTIVE'   THEN 1 END)        AS active_incidents,
    COUNT(CASE WHEN status = 'RESOLVED' THEN 1 END)        AS resolved_incidents,
    COUNT(CASE WHEN status = 'PENDING'  THEN 1 END)        AS pending_incidents,

    -- Incident type breakdown
    COUNT(CASE WHEN incidentId = 'accident'         THEN 1 END)  AS accident_count,
    COUNT(CASE WHEN incidentId = 'pothole'          THEN 1 END)  AS pothole_count,
    COUNT(CASE WHEN incidentId = 'traffic_jam'      THEN 1 END)  AS traffic_jam_count,
    COUNT(CASE WHEN incidentId = 'police_check'     THEN 1 END)  AS police_check_count,
    COUNT(CASE WHEN incidentId = 'roadblock'        THEN 1 END)  AS roadblock_count,
    COUNT(CASE WHEN incidentId = 'toll_plaza_delay' THEN 1 END)  AS toll_plaza_delay_count,
    COUNT(CASE WHEN incidentId = 'vip_movement'     THEN 1 END)  AS vip_movement_count,

    ROUND(COUNT(CASE WHEN status = 'RESOLVED' THEN 1 END)
          * 100.0 / COUNT(*), 2)                           AS resolution_rate_pct,
    CASE
        WHEN COUNT(CASE WHEN status = 'ACTIVE' THEN 1 END)
             >= 5 THEN 'CRITICAL'
        WHEN COUNT(CASE WHEN status = 'ACTIVE' THEN 1 END)
             >= 2 THEN 'HIGH'
        ELSE 'CLEAR'
    END                                                     AS alert_level,
    CURRENT_TIMESTAMP()                                     AS processed_at

FROM smart_city.silver.emergency_clean
GROUP BY area, DATE_TRUNC('hour', timestamp)
ORDER BY hour_window DESC;

num_affected_rows,num_inserted_rows


### Weather Advisory

In [0]:
%sql
CREATE OR REPLACE TABLE smart_city.gold.kpi_weather_advisory AS
SELECT
    area,
    weather_condition,
    DATE_TRUNC('hour', timestamp)                           AS hour_window,
    ROUND(AVG(temperature), 2)                              AS avg_temp_c,
    ROUND(AVG(humidity), 2)                                 AS avg_humidity_pct,
    ROUND(MAX(wind_speed), 2)                               AS max_wind_kmh,
    CASE
        WHEN MAX(wind_speed)    >= 80 
          OR AVG(temperature)   >= 45 
          OR AVG(humidity)      >= 95  THEN 'SEVERE'
        WHEN MAX(wind_speed)    >= 50 
          OR AVG(temperature)   >= 38  THEN 'MODERATE'
        ELSE                                'NORMAL'
    END                                                     AS weather_severity,
    CASE
        WHEN MAX(wind_speed)    >= 80 
          OR AVG(temperature)   >= 45 
          OR AVG(humidity)      >= 95  THEN 'Do not drive'
        WHEN MAX(wind_speed)    >= 50 
          OR AVG(temperature)   >= 38  THEN 'Drive with caution'
        ELSE                                'Safe to drive'
    END                                                     AS driving_advisory,
    CASE
        WHEN weather_condition IN ('Foggy','Stormy','Rainy') THEN 'Poor'
        WHEN weather_condition IN ('Cloudy','Drizzle')       THEN 'Moderate'
        ELSE                                                      'Good'
    END                                                     AS visibility,
    CURRENT_TIMESTAMP()                                     AS processed_at
FROM smart_city.silver.weather_clean
GROUP BY area, weather_condition, DATE_TRUNC('hour', timestamp)
ORDER BY hour_window DESC;

num_affected_rows,num_inserted_rows


### City Health Score


In [0]:
%sql
CREATE OR REPLACE TABLE smart_city.gold.kpi_city_health AS
WITH base AS (
    SELECT
        area,
        DATE_TRUNC('hour', timestamp)                       AS hour_window,
        COUNT(DISTINCT deviceId)                            AS active_vehicles,
        ROUND(AVG(speed), 2)                                AS avg_speed,
        COUNT(CASE WHEN speed > 80 THEN 1 END)              AS overspeeding_count,
        COUNT(*)                                            AS total_events
    FROM smart_city.silver.vehicle_clean
    GROUP BY area, DATE_TRUNC('hour', timestamp)
),
scored AS (
    SELECT *,
        -- traffic score out of 50
        CASE
            WHEN avg_speed <= 40 THEN 50
            WHEN avg_speed <= 60 THEN 40
            WHEN avg_speed <= 80 THEN 30
            ELSE 10
        END                                                 AS traffic_score,
        -- safety score out of 50
        CASE
            WHEN overspeeding_count * 100.0 
                 / total_events <= 10 THEN 50
            WHEN overspeeding_count * 100.0 
                 / total_events <= 30 THEN 35
            ELSE 15
        END                                                 AS safety_score
    FROM base
)
SELECT *,
    traffic_score + safety_score                            AS city_health_score,
    CASE
        WHEN traffic_score + safety_score >= 80 THEN 'Healthy'
        WHEN traffic_score + safety_score >= 60 THEN 'Moderate'
        WHEN traffic_score + safety_score >= 40 THEN 'Stressed'
        ELSE                                         'Critical'
    END                                                     AS city_status,
    CURRENT_TIMESTAMP()                                     AS processed_at
FROM scored
ORDER BY hour_window DESC;

num_affected_rows,num_inserted_rows
